In [2]:
"""
BigAlpha 2026 提交代码 —— ISMP 因子 (聪明钱位置)
=====================================================
"""

import pandas as pd
import numpy as np
import dai


# ═══════════════════════════════════════════════════════════════════════════
# 因子参数
# ═══════════════════════════════════════════════════════════════════════════
VOL_QUANTILE = 0.8       # 高成交量分位数阈值
SMOOTH_WINDOW = 120      # 时序平滑窗口 (日)
MIN_MINUTES = 120        # 每股每日最小分钟数
EPS = 1e-12             # 数值稳定小量

# SMA120 需要 120 交易日历史 ≈ 180 自然日, 给 250 宁多勿少
BUFFER_DAYS = 250


# ═══════════════════════════════════════════════════════════════════════════
# 主函数 (比赛入口)
# ═══════════════════════════════════════════════════════════════════════════

def main(datasources, start_date, end_date):
    """
    比赛要求入口函数。

    Args:
        datasources: dict, 至少包含 {'bar1m': '<分钟量表>'}
        start_date:  str, 评估区间起始时间
        end_date:    str, 评估区间结束时间

    Returns:
        pd.DataFrame, 三列 ['date', 'instrument', 'factor']
    """
    # ✅ 表名从 datasources 取
    bar1m = datasources["bar1m"]

    # ✅ 向前扩查询窗口, 给 SMA120 时序平滑喂历史
    query_start = pd.to_datetime(start_date) - pd.Timedelta(days=BUFFER_DAYS)

    # ═══════════════════════════════════════════════════════════════════
    # 步骤1: DAI SQL 聪明钱位置聚合 (分钟级 → 日级)
    # ═══════════════════════════════════════════════════════════════════
    #
    # 6层CTE:
    #   raw:          分钟基础数据
    #   vol_stats:    GROUP BY 算 vol_thresh (quantile) + n_total
    #   flagged:      JOIN vol_stats, 标记 is_hv
    #   daily_last:   GROUP BY 算 max(date) 找末bar
    #   last_close:   JOIN 取末bar close
    #   agg:          按 (instrument, date) 聚合:
    #                  - hv_pv, hv_vol_sum, hv_count, day_vol_mean, close_last
    #
    # ⚠️ quantile 不用 OVER() 窗口函数, 用 GROUP BY + JOIN 替代 (DuckDB 兼容)
    # ⚠️ max(date)+JOIN 替代 argMax, 确定性取末bar

    sql = f"""
    WITH raw AS (
        SELECT
            instrument,
            CAST(date AS DATE) AS trade_date,
            date,
            close::DOUBLE AS close,
            volume::DOUBLE AS volume
        FROM {bar1m}
        WHERE close > 0
          AND volume > 0
    ),
    vol_stats AS (
        SELECT
            instrument,
            trade_date,
            quantile(volume, {VOL_QUANTILE}) AS vol_thresh,
            count(*) AS n_total
        FROM raw
        GROUP BY instrument, trade_date
    ),
    flagged AS (
        SELECT
            r.instrument,
            r.trade_date,
            r.date,
            r.close,
            r.volume,
            v.vol_thresh,
            v.n_total,
            CASE WHEN r.volume >= v.vol_thresh THEN 1 ELSE 0 END AS is_hv
        FROM raw r
        JOIN vol_stats v
            ON r.instrument = v.instrument
            AND r.trade_date = v.trade_date
    ),
    daily_last AS (
        SELECT
            instrument,
            trade_date,
            max(date) AS last_time
        FROM flagged
        GROUP BY instrument, trade_date
    ),
    last_close AS (
        SELECT
            f.instrument,
            f.trade_date,
            f.close AS close_last
        FROM flagged f
        JOIN daily_last dl
            ON f.instrument = dl.instrument
            AND f.trade_date = dl.trade_date
            AND f.date = dl.last_time
    ),
    agg AS (
        SELECT
            f.instrument,
            f.trade_date,
            sum(CASE WHEN f.is_hv = 1 THEN f.close * f.volume ELSE 0 END) AS hv_pv,
            sum(CASE WHEN f.is_hv = 1 THEN f.volume ELSE 0 END) AS hv_vol_sum,
            sum(CASE WHEN f.is_hv = 1 THEN 1 ELSE 0 END) AS hv_count,
            avg(f.volume) AS day_vol_mean,
            count(*) AS n_bars,
            max(f.n_total) AS n_total,
            lc.close_last
        FROM flagged f
        JOIN last_close lc
            ON f.instrument = lc.instrument
            AND f.trade_date = lc.trade_date
        GROUP BY f.instrument, f.trade_date, lc.close_last
        HAVING max(f.n_total) >= {MIN_MINUTES}
    )
    SELECT
        CAST(a.trade_date AS TIMESTAMP) AS date,
        a.instrument,
        a.hv_pv,
        a.hv_vol_sum,
        a.hv_count,
        a.day_vol_mean,
        a.close_last,
        a.n_bars
    FROM agg a
    ORDER BY a.trade_date, a.instrument
    """

    df = dai.query(
        sql,
        filters={
            'date': [
                query_start.strftime('%Y-%m-%d %H:%M:%S'),
                end_date,
            ]
        },
        compression=True,
    ).df()

    if df.empty:
        return pd.DataFrame(columns=['date', 'instrument', 'factor'])

    # ✅ compression=True 会把 instrument 转成 category, 必须转 string
    df['instrument'] = df['instrument'].astype(str)
    df['date'] = pd.to_datetime(df['date'])

    # ═══════════════════════════════════════════════════════════════════
    # 步骤2: pandas ISMP 公式 + 反转 + SMA 时序平滑
    # ═══════════════════════════════════════════════════════════════════

    df = df.sort_values(['instrument', 'date']).reset_index(drop=True)

    # ✅ 强制转 float: DuckDB 可能返回 Decimal 类型导致运算报错
    for col in ['hv_pv', 'hv_vol_sum', 'hv_count', 'day_vol_mean', 'close_last', 'n_bars']:
        if col in df.columns:
            df[col] = df[col].astype(float)

    # ── HVWAP = Σ(close×volume | is_hv) / Σ(volume | is_hv) ──
    df['ismp_hvvwap'] = np.where(
        df['hv_vol_sum'] > EPS,
        df['hv_pv'] / (df['hv_vol_sum'] + EPS),
        0.0
    )

    # ── ISMP_Score = (close_last - HVWAP) / HVWAP × 100 ──
    df['ismp_score'] = np.where(
        df['ismp_hvvwap'].abs() > EPS,
        (df['close_last'] - df['ismp_hvvwap']) / (df['ismp_hvvwap'] + EPS) * 100.0,
        0.0
    )

    # ── ISMP = ISMP_Score × (hv_vol_mean / day_vol_mean) ──
    # 原代码 hv_vol_mean = mean(volume | is_hv), 这里 hv_vol_sum / hv_count 等价
    hv_vol_mean = np.where(df['hv_count'] > 0, df['hv_vol_sum'] / df['hv_count'], 0.0)
    df['ismp'] = df['ismp_score'] * np.where(
        df['day_vol_mean'] > EPS,
        hv_vol_mean / (df['day_vol_mean'] + EPS),
        0.0
    )

    # ── 清洗 NaN/Inf ──
    for col in ['ismp_hvvwap', 'ismp_score', 'ismp']:
        df[col] = df[col].replace([np.inf, -np.inf], 0.0).fillna(0.0)

    # ── 截面去极值 (0.5% ~ 99.5%) ──
    def winsorize(s):
        lo, hi = s.quantile(0.005), s.quantile(0.995)
        return s.clip(lo, hi)

    df['ismp_clipped'] = df.groupby('date')['ismp'].transform(winsorize)

    # ── 截面 pct rank ──
    df['ismp_rank'] = df.groupby('date')['ismp_clipped'].rank(pct=True)

    # ── 反转 + SMA120 时序平滑 (按股票分组) ──
    # A股: 收盘高于聪明钱成本 → 聪明钱已获利 → 次日反转 → 取反看空
    df['factor'] = df.groupby('instrument')['ismp_rank'].transform(
        lambda x: (-x).rolling(
            window=SMOOTH_WINDOW,
            min_periods=min(SMOOTH_WINDOW, 3),
        ).mean()
    )

    # ═══════════════════════════════════════════════════════════════════
    # 步骤3: 裁回评估区间 + 过滤成分股 + 返回
    # ═══════════════════════════════════════════════════════════════════

    # ✅ 裁回真正的评估区间
    start_ts = pd.to_datetime(start_date)
    end_ts = pd.to_datetime(end_date)
    df = df[(df['date'] >= start_ts) & (df['date'] <= end_ts)]

    # ✅ 过滤到中证1000成分股
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]},
    ).df()
    stk_pool['instrument'] = stk_pool['instrument'].astype(str)
    stk_pool['date'] = pd.to_datetime(stk_pool['date'])

    df = pd.merge(df, stk_pool, how='inner', on=['date', 'instrument'])

    # ── 缺失值处理: 仅 ffill, 严禁 bfill (未来函数) ──
    df = df.sort_values(['instrument', 'date'])
    df['factor'] = df.groupby('instrument')['factor'].ffill()
    df['factor'] = df['factor'].replace([np.inf, -np.inf], np.nan)
    df['factor'] = df['factor'].fillna(0.0)

    return df[['date', 'instrument', 'factor']]


# ═══════════════════════════════════════════════════════════════════════════
# 本地自检代码 (提交前务必跑一遍)
# ═══════════════════════════════════════════════════════════════════════════

def selftest_coverage():
    """自检一: 覆盖度检查"""
    print("=" * 60)
    print("自检一: 覆盖度检查")
    print("=" * 60)

    datasources = {'bar1m': 'bigalpha_2026_stock_bar1m_selftest'}
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-10-31 23:59:59'

    factor_data = main(datasources, start_date, end_date)

    print(f"  总行数: {len(factor_data):,}")
    print(f"  日期数: {factor_data['date'].nunique()}")
    print(f"  股票数: {factor_data['instrument'].nunique()}")
    print(f"  NaN数:  {factor_data['factor'].isna().sum()}")

    for d in ['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05']:
        day_data = factor_data[factor_data['date'] == pd.to_datetime(d)]
        if len(day_data) == 0:
            print(f"  {d}: 无数据")
            continue
        nan_rate = day_data['factor'].isna().mean()
        print(f"  {d}: 缺失率 = {nan_rate:.2%}")
        assert nan_rate < 0.4, f"❌ 覆盖度不足: {d} 缺失率 {nan_rate:.2%}"

    print("  ✅ 覆盖度自检通过\n")


def selftest_lookahead():
    """自检二: 未来函数检查"""
    print("=" * 60)
    print("自检二: 未来函数检查")
    print("=" * 60)

    start_date = '2024-01-01 00:00:00'
    end_date = '2024-01-31 23:59:59'

    df_full = main(
        {'bar1m': 'bigalpha_2026_stock_bar1m_selftest'},
        start_date, end_date,
    )
    df_cut = main(
        {'bar1m': 'bigalpha_2026_stock_bar1m_ahead'},
        start_date, end_date,
    )

    cutoff = '2024-01-31 23:59:59'
    df_full_b = df_full[df_full['date'] <= pd.to_datetime(cutoff)]
    df_cut_b = df_cut[df_cut['date'] <= pd.to_datetime(cutoff)]

    merged = pd.merge(
        df_full_b, df_cut_b,
        on=['date', 'instrument'],
        suffixes=('_full', '_cut'),
    )
    diff = (merged['factor_full'] - merged['factor_cut']).abs()
    bad = merged[diff > 1e-5]

    if len(bad) > 0:
        print(f"  ❌ 发现未来函数, 差异行数: {len(bad)}")
        print(bad.head(10))
    else:
        print("  ✅ 未来函数自检通过")


if __name__ == '__main__':
    selftest_coverage()
    selftest_lookahead()


自检一: 覆盖度检查
